In [3]:
%load_ext autoreload
%autoreload 2


from libthesis import pdf_writer, update_layout

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import pandas
import plotly.express as px
import plotly.graph_objects as go

# Load the data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Filter
df = df[df["kmer_size"] == 25].copy()
df = df[
    df["madb_longest_braid_length"].notnull() &
    df["madb_ungapped_sw_length"].notnull()
].copy()
# Add a symbolic label column for cycles
df["cycle_status"] = df["madb_cycles"].map({True: "has_cycles", False: "no_cycles"})

# Calculate absolute difference from parity
df["parity_diff"] = (df["madb_longest_braid_length"] - df["madb_ungapped_sw_length"]).abs()

# Identify outliers (top 5% furthest from parity)
threshold = 100
df["is_outlier"] = df["parity_diff"] > threshold


# Define fixed mappings
species_colors = {
    "Chimpanzee": "#CC0000",  # Red
    "Gorilla":    "#E69F00",  # Orange-Yellow
    "Macaque":    "#0072B2",  # Blue
}

# Update symbol map to use Plotly-recognized categories
symbol_sequence = {
    "has_cycles": "x",      # open cross
    "no_cycles": "circle"  # open circle
}

# Axis bounds
axis_range = [0, 2000]

# Generate one chart per species
figures = {}
for species in ["Chimpanzee", "Gorilla", "Macaque"]:
    species_df = df[df["species"] == species].copy()
    species_df["is_outlier"] = df.loc[species_df.index, "is_outlier"]
    species_df["symbol"] = species_df["madb_cycles"].map(symbol_sequence)

    fig = px.scatter(
        species_df,
        x="madb_ungapped_sw_length",
        y="madb_longest_braid_length",
        color="species",             # use column name so Plotly handles color properly
        color_discrete_map=species_colors,
        symbol="cycle_status",       # now using named symbolic column
        symbol_map=symbol_sequence,
        hover_data=["unique_id"],
    )
    fig.update_traces(
        marker=dict(size=10, color=species_colors[species]),
        showlegend=False
    )
    fig.add_trace(
        go.Scatter(
            x=axis_range,
            y=axis_range,
            mode="lines",
            line=dict(dash="dot", color="gray"),
            showlegend=False
        )
    )

    # Add outlier labels
    outliers = species_df[species_df["is_outlier"]]
    fig.add_trace(
        go.Scatter(
            x=outliers["madb_ungapped_sw_length"]+50,
            y=outliers["madb_longest_braid_length"],
            text=outliers["unique_id"],
            mode="text",
            textposition="bottom right",
            showlegend=False
        )
    )

    fig.update_layout(
        width=500,
        height=500,
        margin=dict(l=40, r=20, t=20, b=40),
        xaxis_title="Smith-Waterman length (bp)",
        yaxis_title="Longest braid length (bp)",
        xaxis=dict(range=axis_range, scaleanchor="y", scaleratio=1),
        yaxis=dict(range=axis_range),
    )

    figures[species] = fig
    update_layout(fig, in_panel = True)

# Display all
figures["Chimpanzee"].show()
figures["Gorilla"].show()
figures["Macaque"].show()

write_pdf = pdf_writer()
write_pdf(figures["Chimpanzee"], "chimpanzee_longest_braid_ungapped_smith_waterman")

In [5]:
from plotly.subplots import make_subplots

# Create 1-row, 3-column subplot
combined_fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Chimpanzee", "Gorilla", "Macaque"],
    horizontal_spacing=0.05,
    shared_yaxes=True,
    shared_xaxes=True
)

# Add each figure's trace manually into the subplot
for idx, species in enumerate(["Chimpanzee", "Gorilla", "Macaque"], start=1):
    for trace in figures[species].data:
        combined_fig.add_trace(trace, row=1, col=idx)

# Apply layout settings
combined_fig.update_layout(
    width=1500,
    height=500,
    showlegend=False,
    xaxis_title="Ungapped Smith-Waterman length",
    yaxis_title="Longest braid length",
    margin=dict(l=50, r=20, t=40, b=40),
)

# Set all axes to the same scale and bounds
axis_range = [0, 2000]
for i in range(1, 4):
    combined_fig.update_xaxes(range=axis_range, scaleanchor=f"y{i}", row=1, col=i)
    combined_fig.update_yaxes(range=axis_range, row=1, col=i)

combined_fig.show()


In [6]:
import pandas
from scipy.stats import pearsonr

# Load the data
df = pandas.read_csv("../thesis_0.1/df/all_alignment_data.csv")

# Filter to relevant species and non-null values
df = df[
    (df["species"] == "Chimpanzee") &
    (df["kmer_size"] == 25) &
    df["madb_longest_braid_length"].notnull() &
    df["madb_ungapped_sw_length"].notnull()
].copy()

# Compute Pearson correlation
r_value, p_value = pearsonr(df["madb_ungapped_sw_length"], df["madb_longest_braid_length"])

r_value, p_value


(np.float64(0.9697903778177139), np.float64(8.112901626327055e-07))